# PrimeKG → Knowledge Graph (KG) Builder

**Source:** [PrimeKG](https://zitniklab.hms.harvard.edu/projects/PrimeKG/) (`kg.csv`)  
**Species:** *Homo sapiens*

## Relation types processed (25 active)

| # | PrimeKG relation | Output Relation | Head ID | Tail ID |
|---|---|---|---|---|
| 1 | disease_protein | Disease_Gene | DOID/MONDO | NCBI Symbol |
| 2 | pathway_pathway | Pathway_Pathway | Reactome ID | Reactome ID |
| 3 | exposure_protein | Gene_Chemical | NCBI Symbol | PubChem CID |
| 4 | exposure_exposure | ChemicalEntity_ChemicalEntity | PubChem CID | PubChem CID |
| 5 | drug_effect | Phenotype_ChemicalEntity | HPO ID | PubChem/DrugBank |
| 6 | exposure_bioprocess | BiologicalProcess_ChemicalEntity | GO ID | PubChem CID |
| 7 | exposure_molfunc | MolecularFunction_ChemicalEntity | GO ID | PubChem CID |
| 8 | off_label_use | Disease_ChemicalEntity | DOID/MESH | PubChem/DrugBank |
| 9 | pathway_protein | Pathway_Gene | Reactome ID | NCBI Symbol |
| 10 | anatomy_anatomy | Anatomy_Anatomy | UBERON ID | UBERON ID |
| 11 | molfunc_protein | MolFunction_Gene | GO ID | NCBI Symbol |
| 12 | cellcomp_cellcomp | CellularComponent_CellularComponent | GO ID | GO ID |
| 13 | exposure_cellcomp | CellularComponent_ChemicalEntity | GO ID | PubChem CID |
| 14 | bioprocess_protein | BiologicalProcess_Gene | GO ID | NCBI Symbol |
| 15 | anatomy_protein_present | Anatomy_Gene | UBERON ID | NCBI Symbol |
| 16 | protein_protein | Gene_Gene | NCBI Symbol | NCBI Symbol |
| 17 | bioprocess_bioprocess | BiologicalProcess_BiologicalProcess | GO ID | GO ID |
| 18 | phenotype_protein | Phenotype_Gene | HPO ID | NCBI Symbol |
| 19 | disease_phenotype_positive | Phenotype_Disease | HPO ID | DOID/MESH |
| 20 | phenotype_phenotype | Phenotype_Phenotype | HPO ID | HPO ID |
| 21 | drug_protein | Gene_ChemicalEntity | NCBI Symbol | PubChem/DrugBank |
| 22 | disease_disease | Disease_Disease | DOID/MESH | DOID/MESH |
| 23 | drug_drug | ChemicalEntity_ChemicalEntity | PubChem/DrugBank | PubChem/DrugBank |
| 24 | molfunc_molfunc | MolecularFunction_MolecularFunction | GO ID | GO ID |
| 25 | exposure_disease | Disease_ChemicalEntity | DOID/MESH | PubChem CID |
| 26 | cellcomp_protein | CellularComponent_Gene | GO ID | NCBI Symbol |
| 27 | indication | Disease_ChemicalEntity | DOID/MESH | PubChem/DrugBank |

## Dropped (commented out in original)
- `contraindication` — head/tail annotation logic incomplete
- `disease_phenotype_negative` — head/tail annotation logic incomplete
- `anatomy_protein_absent` — head/tail annotation logic incomplete


---
## 0 · Configuration — edit ONLY these two lines

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from io import StringIO

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/"

# ── Derived input paths (do not edit below this line) ────────────────────────
PUBCHEM_PKL_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
PUBCHEM_DB_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
PUBCHEM_SYN_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
PUBCHEM_MESH_PATH = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-MeSH")
MESH2DOID_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MESH_COMB_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
ENS2NCBI_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
DO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
UNIPROT_MAP_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv")
UNIPROT_REC_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
BTO_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/bto/BTO_ALL_COMBINED.csv")
GO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
HMDB_PATH         = os.path.join(BASE_PATH, "databases_for_mapping/hmdb/HMDBoutput.csv")
HPO_PATH          = os.path.join(BASE_PATH, "databases_for_mapping/hpo/HPO.csv")
MONDO_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv")
PRIMEKG_PATH      = os.path.join(BASE_PATH, "primekg/kg.csv")

# Intermediate folder for per-relation CSVs (F1 output / F2 input)
RELWISE_DIR = os.path.join(OUT_PATH, "PrimeKG_relwise/")
os.makedirs(OUT_PATH, exist_ok=True)
os.makedirs(RELWISE_DIR, exist_ok=True)

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/


---
## Part 1 · Split PrimeKG by relation type

Loads the full `kg.csv` and saves one CSV per unique `relation` value into `PrimeKG_relwise/`.

In [2]:
# Load full PrimeKG dataset
KG_file = pd.read_csv(PRIMEKG_PATH)
print(f"PrimeKG loaded: {len(KG_file):,} rows")
print(f"Unique relations: {KG_file['relation'].nunique()}")
print("\nRelation distribution:")
print(KG_file['relation'].value_counts())

/tmp/ipykernel_2667298/1253348248.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  KG_file = pd.read_csv(PRIMEKG_PATH)


PrimeKG loaded: 8,100,498 rows
Unique relations: 30

Relation distribution:
relation
anatomy_protein_present       3036406
drug_drug                     2672628
protein_protein                642150
disease_phenotype_positive     300634
bioprocess_protein             289610
cellcomp_protein               166804
disease_protein                160822
molfunc_protein                139060
drug_effect                    129568
bioprocess_bioprocess          105772
pathway_protein                 85292
disease_disease                 64388
contraindication                61350
drug_protein                    51306
anatomy_protein_absent          39774
phenotype_phenotype             37472
anatomy_anatomy                 28064
molfunc_molfunc                 27148
indication                      18776
cellcomp_cellcomp                9690
phenotype_protein                6660
off-label use                    5136
pathway_pathway                  5070
exposure_disease                 4608
exp

In [3]:
# Entity type counts — useful for scoping
combined_unique_types = pd.concat([KG_file['x_type'], KG_file['y_type']]).unique()
unique_counts = []
for entity_type in combined_unique_types:
    x_count = KG_file.loc[KG_file['x_type'] == entity_type, 'x_name'].nunique()
    y_count = KG_file.loc[KG_file['y_type'] == entity_type, 'y_name'].nunique()
    unique_counts.append({'Entity Type': entity_type, 'Unique Name Count': x_count + y_count})
print("Entity type unique name counts:")
pd.DataFrame(unique_counts).sort_values('Unique Name Count', ascending=False)

Entity type unique name counts:


,Entity Type,Unique Name Count
4,biological_process,57284
0,gene/protein,55342
3,disease,34160
2,effect/phenotype,30622
9,anatomy,28070
5,molecular_function,22338
1,drug,15914
6,cellular_component,8352
8,pathway,4996
7,exposure,1636


In [4]:
# Split and save one file per relation type
for rel in KG_file['relation'].unique():
    subset = KG_file[KG_file['relation'] == rel]
    filename = f"PrimeKG_relwise_{rel.replace(' ', '_')}.csv"
    subset.to_csv(os.path.join(RELWISE_DIR, filename), index=False)

print(f"Saved {KG_file['relation'].nunique()} relation-wise files to: {RELWISE_DIR}")

# Free memory — the raw KG is no longer needed
del KG_file

Saved 30 relation-wise files to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_relwise/


---
## Part 2 · Load all reference dictionaries

In [5]:
# ── 2a. PubChem CID -> IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem
print("PubChem CID dicts loaded")

PubChem CID dicts loaded


In [6]:
# ── 2b. PubChem synonym -> CID and DrugBank -> PubChem CID ───────────────────
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
del Pubchem_Syn_fil

pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB = pubchem[~pubchem['DRUGBANK_ID'].isna()]
DB2PC_dict = dict(zip(pubchem_DB['DRUGBANK_ID'], pubchem_DB['ID']))  # {DB_ID: CID}
del pubchem
print(f"PubChem synonyms: {len(Pubchem_Syn_fil_dict):,} | DrugBank->PubChem: {len(DB2PC_dict):,}")

/tmp/ipykernel_2667298/1635111025.py:8: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


PubChem synonyms: 102,877,691 | DrugBank->PubChem: 10,702


In [7]:
# ── 2c. PubChem CID-MeSH: MeSH name -> PubChem CID ───────────────────────────
# The CID-MeSH file has variable number of tab-separated columns.
# We merge all fields after the first (CID) into a single MeSH name string.
with open(PUBCHEM_MESH_PATH, 'r') as f:
    lines = f.readlines()

fixed_lines = []
for line in lines:
    line = line.rstrip('\n')
    parts = line.split('\t')
    if len(parts) > 2:
        fixed_lines.append(parts[0] + "\t" + " ".join(parts[1:]))
    else:
        fixed_lines.append(line)

mesh_2_Pubchem = pd.read_csv(StringIO("\n".join(fixed_lines)), sep='\t', header=None)
mesh_2_Pubchem_dict = dict(zip(mesh_2_Pubchem[1], mesh_2_Pubchem[0]))  # {MeSH_name: CID}
print(f"MeSH name -> PubChem CID: {len(mesh_2_Pubchem_dict):,}")

MeSH name -> PubChem CID: 93,066


In [8]:
# ── 2d. MeSH -> DOID crosswalk ────────────────────────────────────────────────
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict     = dict(zip(Mesh2DOID['xrefs'],  Mesh2DOID['id']))    # MeSH xref -> DOID
Meshname2DOID_dict = dict(zip(Mesh2DOID['label'],  Mesh2DOID['xrefs'])) # MeSH label -> MeSH xref
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,}")

MeSH -> DOID: 3,687


In [9]:
# ── 2e. MeSH combined and supplementary (ID -> name) ─────────────────────────
MESH = pd.read_csv(MESH_COMB_PATH)
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))
Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
print(f"MeSH combined: {len(MESH_dict):,} | MeSH supplementary: {len(Mesh_supp_dict):,}")

MeSH combined: 38,520 | MeSH supplementary: 324,150


In [10]:
# ── 2f. ENSEMBL <-> NCBI gene crosswalk ──────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [11]:
# ── 2g. NCBI Human gene_info ──────────────────────────────────────────────────
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))
print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")

NCBI Human genes: 193,352


In [12]:
# ── 2h. Gene synonym map — built ONCE globally ────────────────────────────────

# Full NCBI_ALL_GENE scan = expensive. Built once here, used in the helper below.
synonym_map = {}
for _, row in NCBI_ALL_GENE.iterrows():
    for syn in row['Synonyms'].split('|'):
        synonym_map[syn.strip()] = row['Symbol']
print(f"Gene synonym map: {len(synonym_map):,} aliases")

Gene synonym map: 69,564 aliases


In [13]:
# ── 2i. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [14]:
DOID_disname_DOID_dict
DOID_disAltID_dict
DOID_disname_dict

{'DOID:0001816': 'angiosarcoma',
 'DOID:0002116': 'pterygium',
 'DOID:0014667': 'disease of metabolism',
 'DOID:0040001': 'shrimp allergy',
 'DOID:0040002': 'aspirin allergy',
 'DOID:0040003': 'benzylpenicillin allergy',
 'DOID:0040004': 'amoxicillin allergy',
 'DOID:0040005': 'ceftriaxone allergy',
 'DOID:0040006': 'carbamazepine allergy',
 'DOID:0040007': 'abacavir allergy',
 'DOID:0040008': 'isoniazide allergy',
 'DOID:0040009': 'lidocaine allergy',
 'DOID:0040010': 'mepivacaine allergy',
 'DOID:0040011': 'phenobarbital allergy',
 'DOID:0040012': 'phenytoin allergy',
 'DOID:0040013': 'ranitidine allergy',
 'DOID:0040014': 'corticosteroid allergy',
 'DOID:0040015': 'sulfonamide allergy',
 'DOID:0040016': 'sulfamethoxazole allergy',
 'DOID:0040017': 'suprofen allergy',
 'DOID:0040018': 'thiopental allergy',
 'DOID:0040019': 'D-mannitol allergy',
 'DOID:0040020': 'cefotaxime allergy',
 'DOID:0040021': 'cephalosporin allergy',
 'DOID:0040022': 'amodiaquine allergy',
 'DOID:0040023': 'ce

In [15]:
# ── 2j. UniProt ───────────────────────────────────────────────────────────────
Uniprot_Recname = pd.read_csv(UNIPROT_REC_PATH)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
del Uniprot_Recname
print(f"UniProt RecName: {len(Uniprot_Recname_dict):,}")

UniProt RecName: 794,151


In [1]:
# ── 2k. BTO tissue ontology, GO terms, HMDB metabolites, HPO, MONDO ──────────
BTO = pd.read_csv(BTO_PATH)
BTO_dict = dict(zip(BTO['ID'], BTO['name']))

# Only the second (correct) direction is needed for lookups.
GO = pd.read_csv(GO_PATH)
GO_dict              = dict(zip(GO['id'], GO['name']))       # {GO:XXXXXXX: term name}
GO_id_namespace_dict = dict(zip(GO['id'], GO['namespace']))  # {GO:XXXXXXX: namespace}

Metabolite = pd.read_csv(HMDB_PATH)
Metabolite_dict      = dict(zip(Metabolite['accession'], Metabolite['name']))
Metabolite_iupac_dict= dict(zip(Metabolite['accession'], Metabolite['iupac_name']))
del Metabolite

phenotype = pd.read_csv(HPO_PATH)
phenotype = phenotype[phenotype['id'].str.startswith('HP')]
phenotype_dict = dict(zip(phenotype['name'], phenotype['id']))

MONDO = pd.read_csv(MONDO_PATH)
MONDO_dict = dict(zip(MONDO['NAME'], MONDO['ID']))

print(f"BTO: {len(BTO_dict):,} | GO: {len(GO_dict):,} | HPO: {len(phenotype_dict):,} | MONDO: {len(MONDO_dict):,}")

NameError: name 'pd' is not defined

---
## Part 3 · Helper functions

In [17]:
def remove_reversed_duplicates(df, head_col='Head', tail_col='Tail'):
    """
    Remove edges that are duplicated when Head and Tail are swapped.
    For undirected/symmetric relations (e.g. protein_protein), only one of
    (A->B) and (B->A) is kept. Uses sorted tuple as dedup key.
    """
    df = df.copy()
    df['_sorted_pair'] = df.apply(
        lambda row: tuple(sorted([str(row[head_col]), str(row[tail_col])])), axis=1)
    df = df.drop_duplicates(subset='_sorted_pair', keep='last')
    df = df.drop(columns='_sorted_pair')
    return df


def refining_columns(df):
    """
    Rename PrimeKG native columns to EvoAge KG schema column names and
    reorder to a standard set.
    PrimeKG schema: x_name, y_name, relation, display_relation, x_index, x_id, x_type, x_source, y_index, y_id, y_type, y_source
    """
    df = df.rename(columns={
        'x_name':          'Head',
        'y_name':          'Tail',
        'relation':        'Relation',
        'display_relation':'Relation_type'
    })
    desired_columns = [
        'Head', 'Relation', 'Tail', 'Relation_type',
        'x_index', 'x_id', 'x_type', 'x_source',
        'y_index', 'y_id', 'y_type', 'y_source'
    ]
    return df[[c for c in desired_columns if c in df.columns]]


def map_gene_synonyms(df, col='Tail'):
    """
    Normalise a gene column via synonym map -> canonical NCBI Symbol.
    """
    alias_col  = f'{col}_Alias_mapped'
    detail_col = f'{col}_Detail_name'
    ens_col    = f'{col}_ENS'
    df[alias_col]  = df[col].apply(lambda x: synonym_map.get(x, x))
    df[detail_col] = df[alias_col].map(NCBI_ALL_GENEname_dict)
    df[ens_col]    = df[alias_col].map(NCBI_2ENS__dict)
    df[[col, alias_col]] = df[[alias_col, col]]
    return df


def map_Exposure(df, col, mapping_dict):
    """
    Map an exposure/chemical column to PubChem CIDs using a name->CID dict.
    Creates a new column '{col}_Pubchem_ID'; strips trailing .0 from float CIDs.
    """
    new_col = f'{col}_Pubchem_ID'
    df[new_col] = df[col].map(mapping_dict)
    df[new_col] = df[new_col].astype(str).str.replace(r'\.0$', '', regex=True)
    return df


def mapp_Drugbank(df, mapping_dict, col, side):
    """
    Map a DrugBank ID column to PubChem CID (keeps DrugBank ID as fallback).
    Creates '{side}_Pubchem_ID' column; strips trailing .0.
    col: source column in df (e.g. 'y_id' or 'x_id')
    side: 'Head' or 'Tail' — used to name the output column
    """
    new_col = f'{side}_Pubchem_ID'
    df[new_col] = df[col].map(mapping_dict).fillna(df[col])
    df[new_col] = df[new_col].astype(str).str.replace(r'\.0$', '', regex=True)
    return df



def resolve_disease(df, col):
    """
    Resolve disease name -> canonical ID via a 4-source cascade:
      1. DO label -> DOID (preferred)
      2. MeSH label -> MeSH xref (via Meshname2DOID_dict)
      3. MeSH ID -> name (via MESH_dict)
      4. MeSH supplementary ID -> name (via Mesh_supp_dict)
    Swaps resolved ID into col; original name moves to {col}_Orignal.
    Note: steps 2-4 may return MeSH IDs rather than DOIDs where DO coverage is missing.
    """
    df = df.copy()
    id_col   = f'{col}_ID_tmp'
    orig_col = f'{col}_Orignal'
    df[id_col] = (df[col].map(DOID_disname_DOID_dict)
                  .fillna(df[col].map(Meshname2DOID_dict))
                  .fillna(df[col].map(MESH_dict))
                  .fillna(df[col].map(Mesh_supp_dict)))
    before = len(df)
    df = df[df[id_col].notna()]
    print(f"  Disease {col} resolved: {len(df):,} kept / {before - len(df):,} dropped")
    df[orig_col] = df[col]        # save original name
    df[col]      = df[id_col]     # put resolved ID into col
    df.drop(columns=[id_col], inplace=True)
    df[f'{col}_ID_IS'] = 'DOID_ID'
    return df


def select_cols(df, desired):
    return df[[c for c in desired if c in df.columns]]


def save(df, filename):
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=None)
    print(f"Saved {len(df):,} rows -> {path}")


# Standard schema for most outputs
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type", "Source",
    "KG_Source", "Head_ID_IS", "Tail_ID_IS",
    "Head_UBERON_ID", "Head_GO_ID", "Head_HP_ID", "Head_Alias_mapped",
    "Head_Detail_name", "Head_detail_name", "Head_ENS", "Head_DO_name", "Head_DO_Alt_ids",
    "Head_Pubchem_ID", "Head_SMILES_name",
    "Tail_UBERON_ID", "Tail_GO_ID", "Tail_HP_ID", "Tail_Alias_mapped",
    "Tail_Detail_name", "Tail_detail_name", "Tail_ENS", "Tail_DO_name", "Tail_DO_Alt_ids",
    "Tail_Pubchem_ID", "Tail_SMILES_name", "Tail_SMILES_name",
    "evidence", "response", "disease"
]

print("Helper functions defined.")

Helper functions defined.


---
## Part 4 · Load per-relation DataFrames

In [18]:
# Load all relation-wise CSV files saved in Part 1
# Each file is assigned to a variable df_<relation_name>
file_list = glob.glob(os.path.join(RELWISE_DIR, "PrimeKG_relwise_*.csv"))
variable_names = []

for file_path in file_list:
    base_name = os.path.basename(file_path).replace(".csv", "")
    var_name  = base_name.replace("-", "_").replace("PrimeKG_relwise_", "df_")
    globals()[var_name] = pd.read_csv(file_path)
    variable_names.append(var_name)

print(f"Loaded {len(variable_names)} relation DataFrames:")
for n in sorted(variable_names):
    print(f"  {n}")

/tmp/ipykernel_2667298/1770673591.py:9: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)
/tmp/ipykernel_2667298/1770673591.py:9: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)
/tmp/ipykernel_2667298/1770673591.py:9: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)
/tmp/ipykernel_2667298/1770673591.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  globals()[var_name] = pd.read_csv(file_path)


Loaded 30 relation DataFrames:
  df_anatomy_anatomy
  df_anatomy_protein_absent
  df_anatomy_protein_present
  df_bioprocess_bioprocess
  df_bioprocess_protein
  df_cellcomp_cellcomp
  df_cellcomp_protein
  df_contraindication
  df_disease_disease
  df_disease_phenotype_negative
  df_disease_phenotype_positive
  df_disease_protein
  df_drug_drug
  df_drug_effect
  df_drug_protein
  df_exposure_bioprocess
  df_exposure_cellcomp
  df_exposure_disease
  df_exposure_exposure
  df_exposure_molfunc
  df_exposure_protein
  df_indication
  df_molfunc_molfunc
  df_molfunc_protein
  df_off_label_use
  df_pathway_pathway
  df_pathway_protein
  df_phenotype_phenotype
  df_phenotype_protein
  df_protein_protein


In [19]:
df_disease_disease

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,disease_disease,parent-child,35428,2816,disease,adrenal cortex disease,MONDO,38666,4,disease,adrenocortical insufficiency,MONDO
1,disease_disease,parent-child,35429,21034,disease,genetic alopecia,MONDO,38135,5,disease,"alopecia, isolated",MONDO
2,disease_disease,parent-child,35430,2243,disease,hemorrhagic disease,MONDO,38429,9,disease,"inherited bleeding disorder, platelet-type",MONDO
3,disease_disease,parent-child,35431,2245,disease,blood platelet disease,MONDO,38429,9,disease,"inherited bleeding disorder, platelet-type",MONDO
4,disease_disease,parent-child,35432,3847,disease,Mendelian disease,MONDO,38429,9,disease,"inherited bleeding disorder, platelet-type",MONDO
...,...,...,...,...,...,...,...,...,...,...,...,...
64383,disease_disease,parent-child,37694,3308,disease,pleural mesothelioma,MONDO,30725,6292,disease,malignant mesothelioma (disease),MONDO
64384,disease_disease,parent-child,36511,6362,disease,peritoneal mesothelioma (disease),MONDO,30725,6292,disease,malignant mesothelioma (disease),MONDO
64385,disease_disease,parent-child,37343,3669,disease,testicular seminoma,MONDO,33709,2598,disease,germinoma (disease),MONDO
64386,disease_disease,parent-child,37108,3002,disease,dysgerminoma (disease),MONDO,33709,2598,disease,germinoma (disease),MONDO


---
## Part 5 · Process and export each relation type

### 1 · df_disease_protein → Disease_Gene

In [19]:
df_disease_protein = refining_columns(df_disease_protein)
df_disease_protein = remove_reversed_duplicates(df_disease_protein)
df_disease_protein = map_gene_synonyms(df_disease_protein, col='Tail')

# Strip parenthetical notes from disease names (e.g. 'cancer (disease)' -> 'cancer')
df_disease_protein['Head'] = df_disease_protein['Head'].str.replace(r'\s*\(.*?\)', '', regex=True)

df_disease_protein['Head_DO_name'] = (df_disease_protein['Head'].map(DOID_disname_dict)
    .fillna(df_disease_protein['Head'].map(MONDO_dict)))
df_disease_protein['Head_DO_Alt_ids'] = df_disease_protein['Head'].map(DOID_disAltID_dict)

df_disease_protein[['Head', 'Head_DO_name']] = df_disease_protein[['Head_DO_name', 'Head']]

# Drop unmapped rows BEFORE the np.where so Head has no NaN values
df_disease_protein = df_disease_protein[~df_disease_protein['Head'].isna()]
df_disease_protein = df_disease_protein[~df_disease_protein['Tail'].isna()]

# FIX: cast Head to str explicitly so .str accessor works safely
# Use None instead of np.nan to avoid DTypePromotionError in numpy 2.0+
df_disease_protein['Head'] = df_disease_protein['Head'].astype(str)

df_disease_protein['Relation']   = 'Disease_Gene'
df_disease_protein['Head_type']  = 'Disease'
df_disease_protein['Tail_type']  = 'Gene'
df_disease_protein['KG_Source']  = 'PrimeKG'
df_disease_protein['Head_ID_IS'] = np.where(
    df_disease_protein['Head'].str.startswith('MONDO'), 'MONDO',
    np.where(df_disease_protein['Head'].str.startswith('DOID'), 'DOID', None))
df_disease_protein['Tail_ID_IS'] = 'NCBI_Symbol'
df_disease_protein = select_cols(df_disease_protein, DESIRED_COLS)
save(df_disease_protein, 'PrimeKG_Disease_Gene.csv')

Saved 70,821 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Disease_Gene.csv


### 2 · df_pathway_pathway → Pathway_Pathway

In [20]:
df_pathway_pathway = refining_columns(df_pathway_pathway)
df_pathway_pathway = remove_reversed_duplicates(df_pathway_pathway)
# Reactome pathway IDs are in x_id/y_id; swap them into Head/Tail
df_pathway_pathway['Head_detail_name'] = df_pathway_pathway['Head'].copy()
df_pathway_pathway['Tail_detail_name'] = df_pathway_pathway['Tail'].copy()
df_pathway_pathway[['Head', 'x_id']] = df_pathway_pathway[['x_id', 'Head']]
df_pathway_pathway[['Tail', 'y_id']] = df_pathway_pathway[['y_id', 'Tail']]
df_pathway_pathway = df_pathway_pathway.rename(columns={'x_id': 'Head_detail_name', 'y_id': 'Tail_detail_name'})
df_pathway_pathway = df_pathway_pathway[~df_pathway_pathway['Head'].isna()]
df_pathway_pathway = df_pathway_pathway[~df_pathway_pathway['Tail'].isna()]
df_pathway_pathway['Relation']   = 'Pathway_Pathway'
df_pathway_pathway['Head_type']  = 'Pathway'
df_pathway_pathway['Tail_type']  = 'Pathway'
df_pathway_pathway['KG_Source']  = 'PrimeKG'
df_pathway_pathway['Head_ID_IS'] = 'Reactome_Pathway_ID'
df_pathway_pathway['Tail_ID_IS'] = 'Reactome_Pathway_ID'
df_pathway_pathway = select_cols(df_pathway_pathway, DESIRED_COLS)
save(df_pathway_pathway, 'PrimeKG_Pathway_Pathway.csv')

Saved 2,523 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Pathway_Pathway.csv


### 3 · df_exposure_protein → Gene_Chemical (exposure side)

In [21]:
df_exposure_protein = refining_columns(df_exposure_protein)
df_exposure_protein = remove_reversed_duplicates(df_exposure_protein)
# Head is Gene; Tail is chemical (exposure) -> PubChem CID via MeSH name lookup
df_exposure_protein = map_gene_synonyms(df_exposure_protein, col='Head')
df_exposure_protein = map_Exposure(df_exposure_protein, 'Tail', mesh_2_Pubchem_dict)
df_exposure_protein[['Tail_Pubchem_ID', 'Tail']] = df_exposure_protein[['Tail', 'Tail_Pubchem_ID']]
df_exposure_protein['Tail_detail_name'] = df_exposure_protein['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_protein['Tail_SMILES_name'] = df_exposure_protein['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_protein = df_exposure_protein[~df_exposure_protein['Head'].isna()]
df_exposure_protein = df_exposure_protein[~df_exposure_protein['Tail'].isna()]
df_exposure_protein['Relation']   = 'Gene_Chemical'
df_exposure_protein['Head_type']  = 'Gene'
df_exposure_protein['Tail_type']  = 'ChemicalEntity'
df_exposure_protein['KG_Source']  = 'PrimeKG'
df_exposure_protein['Head_ID_IS'] = 'NCBI_Symbol'
df_exposure_protein['Tail_ID_IS'] = 'Pubchem'
df_exposure_protein = select_cols(df_exposure_protein, DESIRED_COLS)
save(df_exposure_protein, 'PrimeKG_Gene_Chemical_1.csv')

Saved 1,212 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Gene_Chemical_1.csv


### 4 · df_exposure_exposure → ChemicalEntity_ChemicalEntity (exposure)

In [22]:
df_exposure_exposure = refining_columns(df_exposure_exposure)
df_exposure_exposure = remove_reversed_duplicates(df_exposure_exposure)
df_exposure_exposure = map_Exposure(df_exposure_exposure, 'Head', mesh_2_Pubchem_dict)
df_exposure_exposure = map_Exposure(df_exposure_exposure, 'Tail', mesh_2_Pubchem_dict)
# Drop rows where PubChem CID could not be resolved
df_exposure_exposure = df_exposure_exposure[df_exposure_exposure['Head_Pubchem_ID'] != 'nan']
df_exposure_exposure = df_exposure_exposure[df_exposure_exposure['Tail_Pubchem_ID'] != 'nan']
df_exposure_exposure[['Head', 'Head_Pubchem_ID']] = df_exposure_exposure[['Head_Pubchem_ID', 'Head']]
df_exposure_exposure[['Tail_Pubchem_ID', 'Tail']] = df_exposure_exposure[['Tail', 'Tail_Pubchem_ID']]
df_exposure_exposure['Head_detail_name'] = df_exposure_exposure['Head'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_exposure['Tail_detail_name'] = df_exposure_exposure['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_exposure['Head_SMILES_name'] = df_exposure_exposure['Head'].map(Pubchem_CID_Smile_Dict)
df_exposure_exposure['Tail_SMILES_name'] = df_exposure_exposure['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_exposure['Relation']   = 'ChemicalEntity_ChemicalEntity'
df_exposure_exposure['Head_type']  = 'ChemicalEntity'
df_exposure_exposure['Tail_type']  = 'ChemicalEntity'
df_exposure_exposure['KG_Source']  = 'PrimeKG'
df_exposure_exposure['Head_ID_IS'] = 'Pubchem'
df_exposure_exposure['Tail_ID_IS'] = 'Pubchem'
df_exposure_exposure = select_cols(df_exposure_exposure, DESIRED_COLS)
save(df_exposure_exposure, 'PrimeKG_Chemical_Chemical_1.csv')

Saved 385 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Chemical_Chemical_1.csv


### 5 · df_drug_effect → Phenotype_ChemicalEntity

In [23]:
df_drug_effect = refining_columns(df_drug_effect)
df_drug_effect

,Head,Relation,Tail,Relation_type,x_index,x_id,x_type,x_source,y_index,y_id,y_type,y_source
0,Levocarnitine,drug_effect,Abdominal pain,side effect,16322,DB00583,drug,DrugBank,23158,2027,effect/phenotype,HPO
1,Levocarnitine,drug_effect,Poor appetite,side effect,16322,DB00583,drug,DrugBank,85849,4396,effect/phenotype,HPO
2,Levocarnitine,drug_effect,Anxiety,side effect,16322,DB00583,drug,DrugBank,22447,739,effect/phenotype,HPO
3,Levocarnitine,drug_effect,Arrhythmia,side effect,16322,DB00583,drug,DrugBank,22831,11675,effect/phenotype,HPO
4,Levocarnitine,drug_effect,Back pain,side effect,16322,DB00583,drug,DrugBank,23469,3418,effect/phenotype,HPO
...,...,...,...,...,...,...,...,...,...,...,...,...
129563,Ileus,drug_effect,Interferon alfa-2b,side effect,25235,2595,effect/phenotype,HPO,15288,DB00105,drug,DrugBank
129564,Dysphonia,drug_effect,Interferon alfa-2b,side effect,24289,1618,effect/phenotype,HPO,15288,DB00105,drug,DrugBank
129565,Coronary artery atherosclerosis,drug_effect,Interferon alfa-2b,side effect,23834,1677,effect/phenotype,HPO,15288,DB00105,drug,DrugBank
129566,Excessive daytime somnolence,drug_effect,Interferon alfa-2b,side effect,84853,1262,effect/phenotype,HPO,15288,DB00105,drug,DrugBank


In [24]:
df_drug_effect = refining_columns(df_drug_effect)
df_drug_effect = remove_reversed_duplicates(df_drug_effect)
df_drug_effect = mapp_Drugbank(df_drug_effect, DB2PC_dict, col='y_id', side='Tail')
# Build HPO ID from x_id (zero-padded to 7 digits with HP: prefix)
df_drug_effect['Head_HP_ID'] = 'HP:' + df_drug_effect['x_id'].astype(str).str.zfill(7)
df_drug_effect[['Head_HP_ID', 'Head']] = df_drug_effect[['Head', 'Head_HP_ID']]
df_drug_effect[['Tail_Pubchem_ID', 'Tail']] = df_drug_effect[['Tail', 'Tail_Pubchem_ID']]
df_drug_effect['Tail_detail_name'] = df_drug_effect['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df_drug_effect['Tail_Pubchem_ID'])
df_drug_effect['Tail_SMILES_name'] = df_drug_effect['Tail'].map(Pubchem_CID_Smile_Dict)
df_drug_effect.rename(columns={'Head_HP_ID': 'Head_detail_name'}, inplace=True)
df_drug_effect['Relation']   = 'Phenotype_ChemicalEntity'
df_drug_effect['Head_type']  = 'Phenotype'
df_drug_effect['Tail_type']  = 'ChemicalEntity'
df_drug_effect['KG_Source']  = 'PrimeKG'
df_drug_effect['Head_ID_IS'] = 'HPO_ID'
# df_drug_effect['Tail_ID_IS'] = np.where(df_drug_effect['Tail'].isna(), np.nan,
#     np.where(df_drug_effect['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))

df_drug_effect['Tail_ID_IS'] = np.where(
    df_drug_effect['Tail'].isna(),
    pd.NA,
    np.where(
        df_drug_effect['Tail'].str.startswith('DB'),
        'Drugbank',
        'Pubchem'
    )
)

df_drug_effect = select_cols(df_drug_effect, DESIRED_COLS)
save(df_drug_effect, 'PrimeKG_Phenotype_Chemical_1.csv')

Saved 64,784 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Phenotype_Chemical_1.csv


### 6 · df_exposure_bioprocess → BiologicalProcess_ChemicalEntity

In [26]:
df_exposure_bioprocess = refining_columns(df_exposure_bioprocess)
df_exposure_bioprocess = remove_reversed_duplicates(df_exposure_bioprocess)
df_exposure_bioprocess = map_Exposure(df_exposure_bioprocess, 'Tail', mesh_2_Pubchem_dict)
# Build GO ID from x_id; fallback to name-based lookup
df_exposure_bioprocess['Head_GO_ID'] = df_exposure_bioprocess['Head'].map(GO_dict).fillna(
    'GO:' + df_exposure_bioprocess['x_id'].astype(str).str.zfill(7))
df_exposure_bioprocess[['Head_GO_ID', 'Head']] = df_exposure_bioprocess[['Head', 'Head_GO_ID']]
df_exposure_bioprocess[['Tail_Pubchem_ID', 'Tail']] = df_exposure_bioprocess[['Tail', 'Tail_Pubchem_ID']]
df_exposure_bioprocess['Tail_detail_name'] = df_exposure_bioprocess['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_bioprocess['Tail_SMILES_name'] = df_exposure_bioprocess['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_bioprocess = df_exposure_bioprocess[~df_exposure_bioprocess['Head'].isna()]
df_exposure_bioprocess = df_exposure_bioprocess[~df_exposure_bioprocess['Tail'].isna()]
df_exposure_bioprocess['Relation']   = 'BiologicalProcess_ChemicalEntity'
df_exposure_bioprocess['Head_type']  = 'BiologicalProcess'
df_exposure_bioprocess['Tail_type']  = 'ChemicalEntity'
df_exposure_bioprocess['KG_Source']  = 'PrimeKG'
df_exposure_bioprocess['Head_ID_IS'] = 'QuickGO'
df_exposure_bioprocess['Tail_ID_IS'] = 'Pubchem'
df_exposure_bioprocess = select_cols(df_exposure_bioprocess, DESIRED_COLS)
save(df_exposure_bioprocess, 'PrimeKG_BiologicalProcess_ChemicalEntity_1.csv')

Saved 1,625 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_BiologicalProcess_ChemicalEntity_1.csv


### 7 · df_exposure_molfunc → MolecularFunction_ChemicalEntity

In [27]:
df_exposure_molfunc = refining_columns(df_exposure_molfunc)
df_exposure_molfunc = remove_reversed_duplicates(df_exposure_molfunc)
df_exposure_molfunc = map_Exposure(df_exposure_molfunc, 'Tail', mesh_2_Pubchem_dict)
df_exposure_molfunc['Head_GO_ID'] = df_exposure_molfunc['Head'].map(GO_dict).fillna(
    'GO:' + df_exposure_molfunc['x_id'].astype(str).str.zfill(7))
df_exposure_molfunc[['Head', 'Head_GO_ID']] = df_exposure_molfunc[['Head_GO_ID', 'Head']]
df_exposure_molfunc[['Tail_Pubchem_ID', 'Tail']] = df_exposure_molfunc[['Tail', 'Tail_Pubchem_ID']]
df_exposure_molfunc['Tail_detail_name'] = df_exposure_molfunc['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_molfunc['Tail_SMILES_name'] = df_exposure_molfunc['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_molfunc['Relation']   = 'MolecularFunction_ChemicalEntity'
df_exposure_molfunc['Head_type']  = 'MolecularFunction'
df_exposure_molfunc['Tail_type']  = 'ChemicalEntity'
df_exposure_molfunc['KG_Source']  = 'PrimeKG'
df_exposure_molfunc['Head_ID_IS'] = 'QuickGO'
df_exposure_molfunc['Tail_ID_IS'] = 'Pubchem'
df_exposure_molfunc = select_cols(df_exposure_molfunc, DESIRED_COLS)
save(df_exposure_molfunc, 'PrimeKG_MolecularFunction_ChemicalEntity1.csv')

Saved 45 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_MolecularFunction_ChemicalEntity1.csv


In [28]:
df_exposure_molfunc

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS,Head_GO_ID,Tail_detail_name,Tail_Pubchem_ID,Tail_SMILES_name,Tail_SMILES_name
45,GO:0019766,MolecularFunction_ChemicalEntity,37034,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,IgA receptor activity,"1,2,4-trichloro-5-(2,4,5-trichlorophenyl)benzene","2,4,5,2',4',5'-hexachlorobiphenyl",C1=C(C(=CC(=C1Cl)Cl)Cl)C2=CC(=C(C=C2Cl)Cl)Cl,C1=C(C(=CC(=C1Cl)Cl)Cl)C2=CC(=C(C=C2Cl)Cl)Cl
46,GO:0019766,MolecularFunction_ChemicalEntity,98491,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,IgA receptor activity,1-chloro-4-[2-chloro-1-(4-chlorophenyl)ethenyl...,Dichlorodiphenyl Dichloroethylene,C1=CC(=CC=C1C(=CCl)C2=CC=C(C=C2)Cl)Cl,C1=CC(=CC=C1C(=CCl)C2=CC=C(C=C2)Cl)Cl
47,GO:0019770,MolecularFunction_ChemicalEntity,37034,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,IgG receptor activity,"1,2,4-trichloro-5-(2,4,5-trichlorophenyl)benzene","2,4,5,2',4',5'-hexachlorobiphenyl",C1=C(C(=CC(=C1Cl)Cl)Cl)C2=CC(=C(C=C2Cl)Cl)Cl,C1=C(C(=CC(=C1Cl)Cl)Cl)C2=CC(=C(C=C2Cl)Cl)Cl
48,GO:0019770,MolecularFunction_ChemicalEntity,98491,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,IgG receptor activity,1-chloro-4-[2-chloro-1-(4-chlorophenyl)ethenyl...,Dichlorodiphenyl Dichloroethylene,C1=CC(=CC=C1C(=CCl)C2=CC=C(C=C2)Cl)Cl,C1=CC(=CC=C1C(=CCl)C2=CC=C(C=C2)Cl)Cl
49,GO:0004104,MolecularFunction_ChemicalEntity,9920327,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,"(1'R,2R,3S,4'S,6S,8'R,10'E,12'S,13'S,14'E,16'E...",abamectin,CC[C@H](C)[C@@H]1[C@H](C=C[C@@]2(O1)C[C@@H]3C[...,CC[C@H](C)[C@@H]1[C@H](C=C[C@@]2(O1)C[C@@H]3C[...
50,GO:0004104,MolecularFunction_ChemicalEntity,5359596,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,arsenic,Arsenic,[As],[As]
51,GO:0004104,MolecularFunction_ChemicalEntity,5463849,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,dicopper;dichloride;trihydroxide,copper oxychloride,[OH-].[OH-].[OH-].[Cl-].[Cl-].[Cu+2].[Cu+2],[OH-].[OH-].[OH-].[Cl-].[Cl-].[Cu+2].[Cu+2]
52,GO:0004104,MolecularFunction_ChemicalEntity,3082,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,2-dimethoxyphosphinothioylsulfanyl-N-methylace...,Dimethoate,CNC(=O)CSP(=S)(OC)OC,CNC(=O)CSP(=S)(OC)OC
53,GO:0004104,MolecularFunction_ChemicalEntity,nan,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,NaN,glyphosate,NaN,NaN
54,GO:0004104,MolecularFunction_ChemicalEntity,nan,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,QuickGO,Pubchem,cholinesterase activity,NaN,Insecticides,NaN,NaN


### 8 · df_off_label_use → Disease_ChemicalEntity (off-label)

In [29]:
df_off_label_use = refining_columns(df_off_label_use)
df_off_label_use = remove_reversed_duplicates(df_off_label_use)
df_off_label_use = mapp_Drugbank(df_off_label_use, DB2PC_dict, col='y_id', side='Tail')
df_off_label_use['Head_DO_name'] = (df_off_label_use['Head'].map(DOID_disname_dict)
    .fillna(df_off_label_use['Head'].map(MONDO_dict)))
df_off_label_use[['Tail_Pubchem_ID', 'Tail']] = df_off_label_use[['Tail', 'Tail_Pubchem_ID']]
df_off_label_use['Tail_detail_name'] = df_off_label_use['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_off_label_use['Tail_SMILES_name'] = df_off_label_use['Tail'].map(Pubchem_CID_Smile_Dict)
df_off_label_use = resolve_disease(df_off_label_use, 'Head')
df_off_label_use['Relation']   = 'Disease_ChemicalEntity'
df_off_label_use['Head_type']  = 'Disease'
df_off_label_use['Tail_type']  = 'ChemicalEntity'
df_off_label_use['KG_Source']  = 'PrimeKG'
df_off_label_use['Tail_ID_IS'] = np.where(df_off_label_use['Tail'].isna(), None,
    np.where(df_off_label_use['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_off_label_use = select_cols(df_off_label_use, DESIRED_COLS)
save(df_off_label_use, 'PrimeKG_Disease_ChemicalEntity1.csv')

  Disease Head resolved: 1,369 kept / 1,199 dropped
Saved 1,369 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Disease_ChemicalEntity1.csv


In [32]:
df_off_label_use

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS,Head_DO_name,Tail_detail_name,Tail_Pubchem_ID,Tail_SMILES_name,Tail_SMILES_name
2569,DOID:10763,Disease_ChemicalEntity,3278,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,NaN,"2-[2,3-dichloro-4-(2-methylidenebutanoyl)pheno...",Etacrynic acid,CCC(=C)C(=O)C1=C(C(=C(C=C1)OCC(=O)O)Cl)Cl,CCC(=C)C(=O)C1=C(C(=C(C=C1)OCC(=O)O)Cl)Cl
2571,DOID:10763,Disease_ChemicalEntity,2471,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,NaN,3-(butylamino)-4-phenoxy-5-sulfamoylbenzoic acid,Bumetanide,CCCCNC1=C(C(=CC(=C1)C(=O)O)S(=O)(=O)N)OC2=CC=C...,CCCCNC1=C(C(=CC(=C1)C(=O)O)S(=O)(=O)N)OC2=CC=C...
2572,DOID:0050425,Disease_ChemicalEntity,6047,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0005391,"(2S)-2-amino-3-(3,4-dihydroxyphenyl)propanoic ...",Levodopa,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O,C1=CC(=C(C=C1C[C@@H](C(=O)O)N)O)O
2573,DOID:0050425,Disease_ChemicalEntity,34359,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0005391,"(2S)-3-(3,4-dihydroxyphenyl)-2-hydrazinyl-2-me...",Carbidopa,C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN,C[C@](CC1=CC(=C(C=C1)O)O)(C(=O)O)NN
2574,DOID:4752,Disease_ChemicalEntity,5745,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0007803,"[2-[(8S,9S,10R,13S,14S,17R)-17-hydroxy-10,13-d...",Cortisone acetate,CC(=O)OCC(=O)[C@]1(CC[C@@H]2[C@@]1(CC(=O)[C@H]...,CC(=O)OCC(=O)[C@]1(CC[C@@H]2[C@@]1(CC(=O)[C@H]...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5130,DOID:11405,Disease_ChemicalEntity,14969,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0005504,"(1S,2R,18R,19R,22S,25R,28R,40S)-48-[(2S,3R,4S,...",Vancomycin,C[C@H]1[C@H]([C@@](C[C@@H](O1)O[C@@H]2[C@H]([C...,C[C@H]1[C@H]([C@@](C[C@@H](O1)O[C@@H]2[C@H]([C...
5132,DOID:2730,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0006541,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...
5133,DOID:4644,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0017610,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...
5134,DOID:11907,Disease_ChemicalEntity,446596,Disease,off-label use,ChemicalEntity,PrimeKG,DOID_ID,Pubchem,MONDO:0001404,"9-[(E)-4-[(2S,3R,4R,5S)-3,4-dihydroxy-5-[[(2S,...",Mupirocin,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...,C[C@H]([C@H]1[C@@H](O1)C[C@H]2CO[C@H]([C@@H]([...


### 9 · df_pathway_protein → Pathway_Gene

In [33]:
df_pathway_protein = refining_columns(df_pathway_protein)
df_pathway_protein = remove_reversed_duplicates(df_pathway_protein)
df_pathway_protein = map_gene_synonyms(df_pathway_protein, col='Tail')
df_pathway_protein.rename(columns={'x_id': 'Head_Detail_name'}, inplace=True)
df_pathway_protein[['Head', 'Head_Detail_name']] = df_pathway_protein[['Head_Detail_name', 'Head']]
df_pathway_protein['Relation']   = 'Pathway_Gene'

df_pathway_protein['Head_type']  = 'Pathway'
df_pathway_protein['Tail_type']  = 'Gene'
df_pathway_protein['KG_Source']  = 'PrimeKG'
df_pathway_protein['Head_ID_IS'] = 'Reactome_ID'
df_pathway_protein['Tail_ID_IS'] = 'NCBI_ID'
df_pathway_protein = select_cols(df_pathway_protein, DESIRED_COLS)
df_pathway_protein

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS,Head_Detail_name,Tail_Alias_mapped,Tail_Detail_name,Tail_ENS
42646,R-HSA-114608,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Platelet degranulation,A1BG,alpha-1-B glycoprotein,ENSG00000121410
42647,R-HSA-6798695,Pathway_Gene,A1BG,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Neutrophil degranulation,A1BG,alpha-1-B glycoprotein,ENSG00000121410
42648,R-HSA-156582,Pathway_Gene,SLC38A1,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Acetylation,NAT2,solute carrier family 38 member 1,ENSG00000111371
42649,R-HSA-74217,Pathway_Gene,ADA,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Purine salvage,ADA,adenosine deaminase,ENSG00000196839
42650,R-HSA-381426,Pathway_Gene,CDH2,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Regulation of Insulin-like Growth Factor (IGF)...,CDH2,cadherin 2,ENSG00000170558
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85287,R-HSA-5576893,Pathway_Gene,KCNE2,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Phase 2 - plateau phase,KCNE2,potassium voltage-gated channel subfamily E re...,ENSG00000159197
85288,R-HSA-3899300,Pathway_Gene,CASP8AP2,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,SUMOylation of transcription cofactors,CASP8AP2,caspase 8 associated protein 2,ENSG00000118412
85289,R-HSA-5628897,Pathway_Gene,SCO2,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,TP53 Regulates Metabolic Genes,SCO2,synthesis of cytochrome C oxidase 2,ENSG00000284194
85290,R-HSA-611105,Pathway_Gene,SCO2,Pathway,interacts with,Gene,PrimeKG,Reactome_ID,NCBI_ID,Respiratory electron transport,SCO2,synthesis of cytochrome C oxidase 2,ENSG00000284194


In [34]:
save(df_pathway_protein, 'PrimeKG_Pathway_Protein_1.csv')

Saved 42,592 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Pathway_Protein_1.csv


### 10 · df_anatomy_anatomy → Anatomy_Anatomy

In [35]:
df_anatomy_anatomy = refining_columns(df_anatomy_anatomy)
df_anatomy_anatomy = remove_reversed_duplicates(df_anatomy_anatomy)
df_anatomy_anatomy['Head_Detail_name'] = 'UBERON:' + df_anatomy_anatomy['x_id'].astype(str).str.zfill(7)
df_anatomy_anatomy['Tail_Detail_name'] = 'UBERON:' + df_anatomy_anatomy['y_id'].astype(str).str.zfill(7)
df_anatomy_anatomy[['Tail', 'Tail_Detail_name']] = df_anatomy_anatomy[['Tail_Detail_name', 'Tail']]
df_anatomy_anatomy[['Head', 'Head_Detail_name']] = df_anatomy_anatomy[['Head_Detail_name', 'Head']]
df_anatomy_anatomy['Relation']   = 'Anatomy_Anatomy'
df_anatomy_anatomy['Head_type']  = 'Anatomy'
df_anatomy_anatomy['Tail_type']  = 'Anatomy'
df_anatomy_anatomy['KG_Source']  = 'PrimeKG'
df_anatomy_anatomy['Head_ID_IS'] = 'UBERON'
df_anatomy_anatomy['Tail_ID_IS'] = 'UBERON'
df_anatomy_anatomy = select_cols(df_anatomy_anatomy, DESIRED_COLS)
save(df_anatomy_anatomy, 'PrimeKG_Anatomy_Anatomy.csv')

Saved 14,032 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Anatomy_Anatomy.csv


### 11 · df_molfunc_protein → MolFunction_Gene

In [36]:
df_molfunc_protein = refining_columns(df_molfunc_protein)
df_molfunc_protein = remove_reversed_duplicates(df_molfunc_protein)
df_molfunc_protein['Head_Detail_name'] = 'GO:' + df_molfunc_protein['x_id'].astype(str).str.zfill(7)
df_molfunc_protein = map_gene_synonyms(df_molfunc_protein, col='Tail')
df_molfunc_protein[['Head_Detail_name', 'Head']] = df_molfunc_protein[['Head', 'Head_Detail_name']]
df_molfunc_protein['Relation']   = 'MolFunction_Gene'
df_molfunc_protein['Head_type']  = 'MolFunction'
df_molfunc_protein['Tail_type']  = 'Gene'
df_molfunc_protein['KG_Source']  = 'PrimeKG'
df_molfunc_protein['Head_ID_IS'] = 'Quick_GO'
df_molfunc_protein['Tail_ID_IS'] = 'NCBI_Symbol'
df_molfunc_protein = select_cols(df_molfunc_protein, DESIRED_COLS)
save(df_molfunc_protein, 'PrimKG_Molfunc_Gene.csv')

Saved 69,530 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimKG_Molfunc_Gene.csv


### 12 · df_cellcomp_cellcomp → CellularComponent_CellularComponent

In [37]:
df_cellcomp_cellcomp = refining_columns(df_cellcomp_cellcomp)
df_cellcomp_cellcomp = remove_reversed_duplicates(df_cellcomp_cellcomp)
df_cellcomp_cellcomp['Head_Detail_name'] = 'GO:' + df_cellcomp_cellcomp['x_id'].astype(str).str.zfill(7)
df_cellcomp_cellcomp['Tail_Detail_name'] = 'GO:' + df_cellcomp_cellcomp['y_id'].astype(str).str.zfill(7)
df_cellcomp_cellcomp[['Head_Detail_name', 'Head']] = df_cellcomp_cellcomp[['Head', 'Head_Detail_name']]
df_cellcomp_cellcomp[['Tail_Detail_name', 'Tail']] = df_cellcomp_cellcomp[['Tail', 'Tail_Detail_name']]
df_cellcomp_cellcomp['Relation']   = 'CellularComponent_CellularComponent'
df_cellcomp_cellcomp['Head_type']  = 'CellularComponent'
df_cellcomp_cellcomp['Tail_type']  = 'CellularComponent'
df_cellcomp_cellcomp['KG_Source']  = 'PrimeKG'
df_cellcomp_cellcomp['Head_ID_IS'] = 'Quick_GO'
df_cellcomp_cellcomp['Tail_ID_IS'] = 'Quick_GO'
df_cellcomp_cellcomp = select_cols(df_cellcomp_cellcomp, DESIRED_COLS)
save(df_cellcomp_cellcomp, 'PrimeKG_cellcomp_cellcomp.csv')

Saved 4,845 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_cellcomp_cellcomp.csv


### 13 · df_exposure_cellcomp → CellularComponent_ChemicalEntity

In [38]:
df_exposure_cellcomp = refining_columns(df_exposure_cellcomp)
df_exposure_cellcomp = remove_reversed_duplicates(df_exposure_cellcomp)
df_exposure_cellcomp['Head_Detail_name'] = 'GO:' + df_exposure_cellcomp['x_id'].astype(str).str.zfill(7)
df_exposure_cellcomp = map_Exposure(df_exposure_cellcomp, 'Tail', mesh_2_Pubchem_dict)
df_exposure_cellcomp[['Head_Detail_name', 'Head']] = df_exposure_cellcomp[['Head', 'Head_Detail_name']]
df_exposure_cellcomp[['Tail_Pubchem_ID', 'Tail']] = df_exposure_cellcomp[['Tail', 'Tail_Pubchem_ID']]
df_exposure_cellcomp = df_exposure_cellcomp[~df_exposure_cellcomp['Tail'].isna()]
df_exposure_cellcomp = df_exposure_cellcomp[df_exposure_cellcomp['Tail'] != 'nan']
df_exposure_cellcomp['Tail_detail_name'] = df_exposure_cellcomp['Tail'].map(Pubchem_IUPAC_CID_Dict)
df_exposure_cellcomp['Tail_SMILES_name'] = df_exposure_cellcomp['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_cellcomp['Relation']   = 'CellularComponent_ChemicalEntity'
df_exposure_cellcomp['Head_type']  = 'CellularComponent'
df_exposure_cellcomp['Tail_type']  = 'ChemicalEntity'
df_exposure_cellcomp['KG_Source']  = 'PrimeKG'
df_exposure_cellcomp['Head_ID_IS'] = 'Quick_GO'
df_exposure_cellcomp['Tail_ID_IS'] = 'Pubchem_ID'
df_exposure_cellcomp = select_cols(df_exposure_cellcomp, DESIRED_COLS)
save(df_exposure_cellcomp, 'PrimeKG_CellComp_Chemical.csv')

Saved 5 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_CellComp_Chemical.csv


### 14 · df_bioprocess_protein → BiologicalProcess_Gene

In [39]:
df_bioprocess_protein = refining_columns(df_bioprocess_protein)
df_bioprocess_protein = remove_reversed_duplicates(df_bioprocess_protein)
df_bioprocess_protein['Head_Detail_name'] = 'GO:' + df_bioprocess_protein['x_id'].astype(str).str.zfill(7)
df_bioprocess_protein = map_gene_synonyms(df_bioprocess_protein, col='Tail')
df_bioprocess_protein[['Head_Detail_name', 'Head']] = df_bioprocess_protein[['Head', 'Head_Detail_name']]
df_bioprocess_protein['Relation']   = 'BiologicalProcess_Gene'
df_bioprocess_protein['Head_type']  = 'BiologicalProcess'
df_bioprocess_protein['Tail_type']  = 'Gene'
df_bioprocess_protein['KG_Source']  = 'PrimeKG'
df_bioprocess_protein['Head_ID_IS'] = 'Quick_GO'
df_bioprocess_protein['Tail_ID_IS'] = 'NCBI_Symbol'
df_bioprocess_protein = select_cols(df_bioprocess_protein, DESIRED_COLS)
save(df_bioprocess_protein, 'PrimeKG_BiologicalProcess_Gene.csv')

Saved 144,805 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_BiologicalProcess_Gene.csv


### 15 · df_anatomy_protein_present → Anatomy_Gene

In [40]:
df_anatomy_protein_present = refining_columns(df_anatomy_protein_present)
df_anatomy_protein_present = remove_reversed_duplicates(df_anatomy_protein_present)
df_anatomy_protein_present['Head_Detail_name'] = 'UBERON:' + df_anatomy_protein_present['x_id'].astype(str).str.zfill(7)
df_anatomy_protein_present = map_gene_synonyms(df_anatomy_protein_present, col='Tail')
df_anatomy_protein_present[['Head', 'Head_Detail_name']] = df_anatomy_protein_present[['Head_Detail_name', 'Head']]
df_anatomy_protein_present['Relation']   = 'Anatomy_Gene'
df_anatomy_protein_present['Head_type']  = 'Anatomy'
df_anatomy_protein_present['Tail_type']  = 'Gene'
df_anatomy_protein_present['KG_Source']  = 'PrimeKG'
df_anatomy_protein_present['Head_ID_IS'] = 'UBERON_ID'
df_anatomy_protein_present['Tail_ID_IS'] = 'NCBI_Symbol'
df_anatomy_protein_present = select_cols(df_anatomy_protein_present, DESIRED_COLS)
save(df_anatomy_protein_present, 'PriemKG_Anatomy_Gene.csv')

Saved 1,518,203 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PriemKG_Anatomy_Gene.csv


### 16 · df_protein_protein → Gene_Gene

In [41]:
df_protein_protein = refining_columns(df_protein_protein)
df_protein_protein = remove_reversed_duplicates(df_protein_protein)
df_protein_protein = map_gene_synonyms(df_protein_protein, col='Head')
df_protein_protein = map_gene_synonyms(df_protein_protein, col='Tail')
df_protein_protein['Relation']   = 'Gene_Gene'
df_protein_protein['Head_type']  = 'Gene'
df_protein_protein['Tail_type']  = 'Gene'
df_protein_protein['KG_Source']  = 'PrimeKG'
df_protein_protein['Head_ID_IS'] = 'NCBI_Symbol'
df_protein_protein['Tail_ID_IS'] = 'NCBI_Symbol'
df_protein_protein = select_cols(df_protein_protein, DESIRED_COLS)
save(df_protein_protein, 'PrimeKG_Gene_Gene.csv')

Saved 321,075 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Gene_Gene.csv


### 17 · df_bioprocess_bioprocess → BiologicalProcess_BiologicalProcess

In [42]:
df_bioprocess_bioprocess = refining_columns(df_bioprocess_bioprocess)
df_bioprocess_bioprocess = remove_reversed_duplicates(df_bioprocess_bioprocess)
df_bioprocess_bioprocess['Head_Detail_name'] = 'GO:' + df_bioprocess_bioprocess['x_id'].astype(str).str.zfill(7)
df_bioprocess_bioprocess['Tail_Detail_name'] = 'GO:' + df_bioprocess_bioprocess['y_id'].astype(str).str.zfill(7)
df_bioprocess_bioprocess[['Head', 'Head_Detail_name']] = df_bioprocess_bioprocess[['Head_Detail_name', 'Head']]
df_bioprocess_bioprocess[['Tail', 'Tail_Detail_name']] = df_bioprocess_bioprocess[['Tail_Detail_name', 'Tail']]
df_bioprocess_bioprocess['Relation']   = 'BiologicalProcess_BiologicalProcess'
df_bioprocess_bioprocess['Head_type']  = 'BiologicalProcess'
df_bioprocess_bioprocess['Tail_type']  = 'BiologicalProcess'
df_bioprocess_bioprocess['KG_Source']  = 'PrimeKG'
df_bioprocess_bioprocess['Head_ID_IS'] = 'Quick_GO'
df_bioprocess_bioprocess['Tail_ID_IS'] = 'Quick_GO'
df_bioprocess_bioprocess = select_cols(df_bioprocess_bioprocess, DESIRED_COLS)
save(df_bioprocess_bioprocess, 'PrimeKG_BiologicalProcess_BiologicalProcess.csv')

Saved 52,886 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_BiologicalProcess_BiologicalProcess.csv


### 18 · df_phenotype_protein → Phenotype_Gene

In [43]:
df_phenotype_protein = refining_columns(df_phenotype_protein)
df_phenotype_protein = remove_reversed_duplicates(df_phenotype_protein)
df_phenotype_protein['Head_Detail_name'] = 'HP:' + df_phenotype_protein['x_id'].astype(str).str.zfill(7)
df_phenotype_protein = map_gene_synonyms(df_phenotype_protein, col='Tail')
df_phenotype_protein[['Head', 'Head_Detail_name']] = df_phenotype_protein[['Head_Detail_name', 'Head']]
df_phenotype_protein['Relation']   = 'Phenotype_Gene'
df_phenotype_protein['Head_type']  = 'Phenotype'
df_phenotype_protein['Tail_type']  = 'Gene'
df_phenotype_protein['KG_Source']  = 'PrimeKG'
df_phenotype_protein['Head_ID_IS'] = 'HPO_ID'
df_phenotype_protein['Tail_ID_IS'] = 'NCBI_ID'
df_phenotype_protein = select_cols(df_phenotype_protein, DESIRED_COLS)
save(df_phenotype_protein, 'PrimeKG_Phenotype_Gene.csv')

Saved 3,330 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Phenotype_Gene.csv


### 19 · df_disease_phenotype_positive → Phenotype_Disease

In [44]:
df_disease_phenotype_positive = refining_columns(df_disease_phenotype_positive)
df_disease_phenotype_positive = remove_reversed_duplicates(df_disease_phenotype_positive)
df_disease_phenotype_positive['Head_Detail_name'] = 'HP:' + df_disease_phenotype_positive['x_id'].astype(str).str.zfill(7)
df_disease_phenotype_positive[['Head', 'Head_Detail_name']] = df_disease_phenotype_positive[['Head_Detail_name', 'Head']]
# Remove rows with '_' in Head (malformed HPO IDs)
df_disease_phenotype_positive = df_disease_phenotype_positive[~df_disease_phenotype_positive['Head'].str.contains('_')]
df_disease_phenotype_positive = resolve_disease(df_disease_phenotype_positive, 'Tail')
df_disease_phenotype_positive['Relation']   = 'Phenotype_Disease'
df_disease_phenotype_positive['Head_type']  = 'Phenotype'
df_disease_phenotype_positive['Tail_type']  = 'Disease'
df_disease_phenotype_positive['KG_Source']  = 'PrimeKG'
df_disease_phenotype_positive['Head_ID_IS'] = 'HPO_ID'
df_disease_phenotype_positive = select_cols(df_disease_phenotype_positive, DESIRED_COLS)
save(df_disease_phenotype_positive, 'PrimeKG_Phenotype_Disease_1.csv')

  Disease Tail resolved: 51,887 kept / 98,351 dropped
Saved 51,887 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Phenotype_Disease_1.csv


### 20 · df_phenotype_phenotype → Phenotype_Phenotype

In [45]:
df_phenotype_phenotype = refining_columns(df_phenotype_phenotype)
df_phenotype_phenotype = remove_reversed_duplicates(df_phenotype_phenotype)
df_phenotype_phenotype['Head_Detail_name'] = 'HP:' + df_phenotype_phenotype['x_id'].astype(str).str.zfill(7)
df_phenotype_phenotype['Tail_Detail_name'] = 'HP:' + df_phenotype_phenotype['y_id'].astype(str).str.zfill(7)
df_phenotype_phenotype[['Head_Detail_name', 'Head']] = df_phenotype_phenotype[['Head', 'Head_Detail_name']]
df_phenotype_phenotype[['Tail_Detail_name', 'Tail']] = df_phenotype_phenotype[['Tail', 'Tail_Detail_name']]
df_phenotype_phenotype['Relation']   = 'Phenotype_Phenotype'
df_phenotype_phenotype['Head_type']  = 'Phenotype'
df_phenotype_phenotype['Tail_type']  = 'Phenotype'
df_phenotype_phenotype['KG_Source']  = 'PrimeKG'
df_phenotype_phenotype['Head_ID_IS'] = 'HPO_ID'
df_phenotype_phenotype['Tail_ID_IS'] = 'HPO_ID'
df_phenotype_phenotype = select_cols(df_phenotype_phenotype, DESIRED_COLS)
save(df_phenotype_phenotype, 'PrimeKG_Phenotype_Phenotype.csv')

Saved 18,736 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Phenotype_Phenotype.csv


### 21 · df_drug_protein → Gene_ChemicalEntity

In [47]:
df_drug_protein = refining_columns(df_drug_protein)
df_drug_protein = remove_reversed_duplicates(df_drug_protein)
df_drug_protein = mapp_Drugbank(df_drug_protein, DB2PC_dict, col='y_id', side='Tail')
df_drug_protein = map_gene_synonyms(df_drug_protein, col='Head')
df_drug_protein[['Tail', 'Tail_Pubchem_ID']] = df_drug_protein[['Tail_Pubchem_ID', 'Tail']]
df_drug_protein['Tail_detail_name'] = df_drug_protein['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df_drug_protein['Tail_Pubchem_ID'])
df_drug_protein['Tail_SMILES_name'] = df_drug_protein['Tail'].map(Pubchem_CID_Smile_Dict)
df_drug_protein['Relation']   = 'Gene_ChemicalEntity'
df_drug_protein['Head_type']  = 'Gene'
df_drug_protein['Tail_type']  = 'ChemicalEntity'
df_drug_protein['KG_Source']  = 'PrimeKG'
df_drug_protein['Head_ID_IS'] = 'NCBI_Symbol'
df_drug_protein['Tail_ID_IS'] = np.where(df_drug_protein['Tail_Pubchem_ID'].isna(), None,
    np.where(df_drug_protein['Tail_Pubchem_ID'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_drug_protein = select_cols(df_drug_protein, DESIRED_COLS)
save(df_drug_protein, 'PrimeKG_Gene_ChemicalEntity.csv')

Saved 25,312 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Gene_ChemicalEntity.csv


### 22 · df_disease_disease → Disease_Disease

In [52]:
df_disease_disease = refining_columns(df_disease_disease)
df_disease_disease

,Head,Relation,Tail,Relation_type,x_index,x_id,x_type,x_source,y_index,y_id,y_type,y_source
0,adrenal cortex disease,disease_disease,adrenocortical insufficiency,parent-child,35428,2816,disease,MONDO,38666,4,disease,MONDO
1,genetic alopecia,disease_disease,"alopecia, isolated",parent-child,35429,21034,disease,MONDO,38135,5,disease,MONDO
2,hemorrhagic disease,disease_disease,"inherited bleeding disorder, platelet-type",parent-child,35430,2243,disease,MONDO,38429,9,disease,MONDO
3,blood platelet disease,disease_disease,"inherited bleeding disorder, platelet-type",parent-child,35431,2245,disease,MONDO,38429,9,disease,MONDO
4,Mendelian disease,disease_disease,"inherited bleeding disorder, platelet-type",parent-child,35432,3847,disease,MONDO,38429,9,disease,MONDO
...,...,...,...,...,...,...,...,...,...,...,...,...
64383,pleural mesothelioma,disease_disease,malignant mesothelioma (disease),parent-child,37694,3308,disease,MONDO,30725,6292,disease,MONDO
64384,peritoneal mesothelioma (disease),disease_disease,malignant mesothelioma (disease),parent-child,36511,6362,disease,MONDO,30725,6292,disease,MONDO
64385,testicular seminoma,disease_disease,germinoma (disease),parent-child,37343,3669,disease,MONDO,33709,2598,disease,MONDO
64386,dysgerminoma (disease),disease_disease,germinoma (disease),parent-child,37108,3002,disease,MONDO,33709,2598,disease,MONDO


In [53]:
df_disease_disease = remove_reversed_duplicates(df_disease_disease)
df_disease_disease['Head'] = df_disease_disease['Head'].str.replace(r'\s*\(.*?\)', '', regex=True)
df_disease_disease['Tail'] = df_disease_disease['Tail'].str.replace(r'\s*\(.*?\)', '', regex=True)
df_disease_disease = resolve_disease(df_disease_disease, 'Head')
df_disease_disease = resolve_disease(df_disease_disease, 'Tail')
df_disease_disease

  Disease Head resolved: 12,069 kept / 20,125 dropped
  Disease Tail resolved: 5,316 kept / 6,753 dropped


,Head,Relation,Tail,Relation_type,x_index,x_id,x_type,x_source,y_index,y_id,y_type,y_source,Head_Orignal,Head_ID_IS,Tail_Orignal,Tail_ID_IS
89,DOID:0080637,disease_disease,DOID:10629,parent-child,27499,12409_12605_13293_12604_13377_9631_14050_13130_62,disease,MONDO_grouped,27916,12709_13376_14059_14635_12408_13783_7996_9630_...,disease,MONDO_grouped,isolated microphthalmia,DOID_ID,microphthalmia,DOID_ID
200,DOID:13564,disease_disease,DOID:0050153,parent-child,33216,5657_240,disease,MONDO_grouped,35528,266,disease,MONDO,aspergillosis,DOID_ID,pulmonary aspergilloma,DOID_ID
2818,DOID:255,disease_disease,DOID:2725,parent-child,33562,6500_3096_2323,disease,MONDO_grouped,36596,2407,disease,MONDO,hemangioma,DOID_ID,capillary hemangioma,DOID_ID
3833,DOID:255,disease_disease,DOID:471,parent-child,33562,6500_3096_2323,disease,MONDO_grouped,36444,3110,disease,MONDO,hemangioma,DOID_ID,skin hemangioma,DOID_ID
3863,DOID:2921,disease_disease,DOID:4778,parent-child,33678,2462_3137_1645_1871_3135_3138,disease,MONDO_grouped,36319,3134_1644_3139,disease,MONDO_grouped,glomerulonephritis,DOID_ID,proliferative glomerulonephritis,DOID_ID
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64375,DOID:9952,disease_disease,DOID:12603,parent-child,33621,4967,disease,MONDO,28465,10643,disease,MONDO,acute lymphoblastic leukemia,DOID_ID,acute leukemia,DOID_ID
64378,DOID:0050585,disease_disease,DOID:811,parent-child,28482,13225_12923_12071_10020_6536,disease,MONDO_grouped,33711,6573,disease,MONDO,congenital generalized lipodystrophy,DOID_ID,lipodystrophy,DOID_ID
64380,DOID:11406,disease_disease,DOID:13141,parent-child,39366,1280,disease,MONDO,33654,20283_6651,disease,MONDO_grouped,choroiditis,DOID_ID,uveitis,DOID_ID
64384,DOID:1788,disease_disease,DOID:1790,parent-child,36511,6362,disease,MONDO,30725,6292,disease,MONDO,peritoneal mesothelioma,DOID_ID,malignant mesothelioma,DOID_ID


In [57]:
df_disease_disease['Head_Detail_name'] = df_disease_disease['Head'].map(DOID_disname_dict)
df_disease_disease['Tail_Detail_name'] = df_disease_disease['Tail'].map(DOID_disname_dict)
df_disease_disease

,Head,Relation,Tail,Relation_type,x_index,x_id,x_type,x_source,y_index,y_id,y_type,y_source,Head_Orignal,Head_ID_IS,Tail_Orignal,Tail_ID_IS,Head_Detail_name,Tail_Detail_name
89,DOID:0080637,disease_disease,DOID:10629,parent-child,27499,12409_12605_13293_12604_13377_9631_14050_13130_62,disease,MONDO_grouped,27916,12709_13376_14059_14635_12408_13783_7996_9630_...,disease,MONDO_grouped,isolated microphthalmia,DOID_ID,microphthalmia,DOID_ID,isolated microphthalmia,microphthalmia
200,DOID:13564,disease_disease,DOID:0050153,parent-child,33216,5657_240,disease,MONDO_grouped,35528,266,disease,MONDO,aspergillosis,DOID_ID,pulmonary aspergilloma,DOID_ID,aspergillosis,pulmonary aspergilloma
2818,DOID:255,disease_disease,DOID:2725,parent-child,33562,6500_3096_2323,disease,MONDO_grouped,36596,2407,disease,MONDO,hemangioma,DOID_ID,capillary hemangioma,DOID_ID,hemangioma,capillary hemangioma
3833,DOID:255,disease_disease,DOID:471,parent-child,33562,6500_3096_2323,disease,MONDO_grouped,36444,3110,disease,MONDO,hemangioma,DOID_ID,skin hemangioma,DOID_ID,hemangioma,skin hemangioma
3863,DOID:2921,disease_disease,DOID:4778,parent-child,33678,2462_3137_1645_1871_3135_3138,disease,MONDO_grouped,36319,3134_1644_3139,disease,MONDO_grouped,glomerulonephritis,DOID_ID,proliferative glomerulonephritis,DOID_ID,glomerulonephritis,proliferative glomerulonephritis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64375,DOID:9952,disease_disease,DOID:12603,parent-child,33621,4967,disease,MONDO,28465,10643,disease,MONDO,acute lymphoblastic leukemia,DOID_ID,acute leukemia,DOID_ID,acute lymphoblastic leukemia,acute leukemia
64378,DOID:0050585,disease_disease,DOID:811,parent-child,28482,13225_12923_12071_10020_6536,disease,MONDO_grouped,33711,6573,disease,MONDO,congenital generalized lipodystrophy,DOID_ID,lipodystrophy,DOID_ID,congenital generalized lipodystrophy,lipodystrophy
64380,DOID:11406,disease_disease,DOID:13141,parent-child,39366,1280,disease,MONDO,33654,20283_6651,disease,MONDO_grouped,choroiditis,DOID_ID,uveitis,DOID_ID,choroiditis,uveitis
64384,DOID:1788,disease_disease,DOID:1790,parent-child,36511,6362,disease,MONDO,30725,6292,disease,MONDO,peritoneal mesothelioma,DOID_ID,malignant mesothelioma,DOID_ID,peritoneal mesothelioma,malignant mesothelioma


In [58]:
df_disease_disease = df_disease_disease[~df_disease_disease['Head_Detail_name'].isna()]
df_disease_disease = df_disease_disease[~df_disease_disease['Tail_Detail_name'].isna()]
df_disease_disease['Relation']   = 'Disease_Disease'
df_disease_disease['Head_type']  = 'Disease'
df_disease_disease['Tail_type']  = 'Disease'
df_disease_disease['KG_Source']  = 'PrimeKG'
df_disease_disease = select_cols(df_disease_disease, DESIRED_COLS)
save(df_disease_disease, 'PrimeKG_Disease_Disease.csv')

Saved 5,316 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Disease_Disease.csv


### 23 · df_drug_drug → ChemicalEntity_ChemicalEntity (drug-drug)

In [59]:
df_drug_drug = refining_columns(df_drug_drug)
df_drug_drug = remove_reversed_duplicates(df_drug_drug)
df_drug_drug = mapp_Drugbank(df_drug_drug, DB2PC_dict, col='y_id', side='Tail')
df_drug_drug = mapp_Drugbank(df_drug_drug, DB2PC_dict, col='x_id', side='Head')
df_drug_drug[['Head', 'Head_Pubchem_ID']] = df_drug_drug[['Head_Pubchem_ID', 'Head']]
df_drug_drug[['Tail', 'Tail_Pubchem_ID']] = df_drug_drug[['Tail_Pubchem_ID', 'Tail']]
df_drug_drug['Head_detail_name'] = df_drug_drug['Head'].map(Pubchem_IUPAC_CID_Dict).fillna(df_drug_drug['Head_Pubchem_ID'])
df_drug_drug['Tail_detail_name'] = df_drug_drug['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df_drug_drug['Tail_Pubchem_ID'])
df_drug_drug['Head_SMILES_name'] = df_drug_drug['Head'].map(Pubchem_CID_Smile_Dict)
df_drug_drug['Tail_SMILES_name'] = df_drug_drug['Tail'].map(Pubchem_CID_Smile_Dict)
df_drug_drug['Relation']   = 'ChemicalEntity_ChemicalEntity'
df_drug_drug['Head_type']  = 'ChemicalEntity'
df_drug_drug['Tail_type']  = 'ChemicalEntity'
df_drug_drug['KG_Source']  = 'PrimeKG'
df_drug_drug['Head_ID_IS'] = np.where(df_drug_drug['Head'].isna(), None,
    np.where(df_drug_drug['Head'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_drug_drug['Tail_ID_IS'] = np.where(df_drug_drug['Tail'].isna(), None,
    np.where(df_drug_drug['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_drug_drug = select_cols(df_drug_drug, DESIRED_COLS)
save(df_drug_drug, 'PrimeKG_Chemical_Chemical_2.csv')

Saved 1,336,314 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Chemical_Chemical_2.csv


### 24 · df_molfunc_molfunc → MolecularFunction_MolecularFunction

In [60]:
df_molfunc_molfunc = refining_columns(df_molfunc_molfunc)
df_molfunc_molfunc = remove_reversed_duplicates(df_molfunc_molfunc)
df_molfunc_molfunc['Head_Detail_name'] = 'GO:' + df_molfunc_molfunc['x_id'].astype(str).str.zfill(7)
df_molfunc_molfunc['Tail_Detail_name'] = 'GO:' + df_molfunc_molfunc['y_id'].astype(str).str.zfill(7)
df_molfunc_molfunc[['Head_Detail_name', 'Head']] = df_molfunc_molfunc[['Head', 'Head_Detail_name']]
df_molfunc_molfunc[['Tail_Detail_name', 'Tail']] = df_molfunc_molfunc[['Tail', 'Tail_Detail_name']]
df_molfunc_molfunc['Relation']   = 'MolecularFunction_MolecularFunction'
df_molfunc_molfunc['Head_type']  = 'MolecularFunction'
df_molfunc_molfunc['Tail_type']  = 'MolecularFunction'
df_molfunc_molfunc['KG_Source']  = 'PrimeKG'
df_molfunc_molfunc['Head_ID_IS'] = 'Quick_GO'
df_molfunc_molfunc['Tail_ID_IS'] = 'Quick_GO'
df_molfunc_molfunc = select_cols(df_molfunc_molfunc, DESIRED_COLS)
save(df_molfunc_molfunc, 'PrimekG_MolFxn_MolFxn.csv')

Saved 13,574 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimekG_MolFxn_MolFxn.csv


### 25 · df_exposure_disease → Disease_ChemicalEntity (exposure)

In [61]:
df_exposure_disease = refining_columns(df_exposure_disease)
df_exposure_disease = remove_reversed_duplicates(df_exposure_disease)
df_exposure_disease['Head'] = df_exposure_disease['Head'].str.replace(r'\s*\(.*?\)', '', regex=True)
df_exposure_disease = resolve_disease(df_exposure_disease, 'Head')
df_exposure_disease = map_Exposure(df_exposure_disease, 'Tail', mesh_2_Pubchem_dict)
df_exposure_disease[['Tail_Pubchem_ID', 'Tail']] = df_exposure_disease[['Tail', 'Tail_Pubchem_ID']]
df_exposure_disease['Tail_detail_name'] = df_exposure_disease['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df_exposure_disease['Tail_Pubchem_ID'])
df_exposure_disease['Tail_SMILES_name'] = df_exposure_disease['Tail'].map(Pubchem_CID_Smile_Dict)
df_exposure_disease['Relation']   = 'Disease_ChemicalEntity'
df_exposure_disease['Head_type']  = 'Disease'
df_exposure_disease['Tail_type']  = 'ChemicalEntity'
df_exposure_disease['KG_Source']  = 'PrimeKG'


  Disease Head resolved: 1,377 kept / 927 dropped


DTypePromotionError: The DType <class 'numpy.dtypes._PyFloatDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes._PyFloatDType'>, <class 'numpy.dtypes.StrDType'>)

In [62]:
df_exposure_disease['Tail_ID_IS'] = np.where(df_exposure_disease['Tail'].isna(), None,
    np.where(df_exposure_disease['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_exposure_disease = select_cols(df_exposure_disease, DESIRED_COLS)
save(df_exposure_disease, 'PrimeKG_Disease_Chemical.csv')

Saved 1,377 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Disease_Chemical.csv


### 26 · df_cellcomp_protein → CellularComponent_Gene

In [63]:
df_cellcomp_protein = refining_columns(df_cellcomp_protein)
df_cellcomp_protein = remove_reversed_duplicates(df_cellcomp_protein)
df_cellcomp_protein['Head_Detail_name'] = 'GO:' + df_cellcomp_protein['x_id'].astype(str).str.zfill(7)
df_cellcomp_protein = map_gene_synonyms(df_cellcomp_protein, col='Tail')
df_cellcomp_protein[['Head', 'Head_Detail_name']] = df_cellcomp_protein[['Head_Detail_name', 'Head']]
df_cellcomp_protein['Relation']   = 'CellularComponent_Gene'
df_cellcomp_protein['Head_type']  = 'CellularComponent'
df_cellcomp_protein['Tail_type']  = 'Gene'
df_cellcomp_protein['KG_Source']  = 'PrimeKG'
df_cellcomp_protein['Head_ID_IS'] = 'Quick_GO'
df_cellcomp_protein['Tail_ID_IS'] = 'NCBI_ID'
df_cellcomp_protein = select_cols(df_cellcomp_protein, DESIRED_COLS)
save(df_cellcomp_protein, 'PrimekG_CellularComp_Gene.csv')

Saved 83,402 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimekG_CellularComp_Gene.csv


### 27 · df_indication → Disease_ChemicalEntity (indication)

In [64]:
df_indication = refining_columns(df_indication)
df_indication = remove_reversed_duplicates(df_indication)
df_indication['Head'] = df_indication['Head'].str.replace(r'\s*\(.*?\)', '', regex=True)
df_indication = resolve_disease(df_indication, 'Head')
df_indication = mapp_Drugbank(df_indication, DB2PC_dict, col='y_id', side='Tail')
df_indication[['Tail', 'Tail_Pubchem_ID']] = df_indication[['Tail_Pubchem_ID', 'Tail']]
df_indication['Tail_detail_name'] = df_indication['Tail'].map(Pubchem_IUPAC_CID_Dict).fillna(df_indication['Tail_Pubchem_ID'])
df_indication['Tail_SMILES_name'] = df_indication['Tail'].map(Pubchem_CID_Smile_Dict)
df_indication['Relation']   = 'Disease_ChemicalEntity'
df_indication['Head_type']  = 'Disease'
df_indication['Tail_type']  = 'ChemicalEntity'
df_indication['KG_Source']  = 'PrimeKG'
df_indication['Tail_ID_IS'] = np.where(df_indication['Tail'].isna(), None,
    np.where(df_indication['Tail'].str.startswith('DB'), 'Drugbank', 'Pubchem'))
df_indication = select_cols(df_indication, DESIRED_COLS)
save(df_indication, 'PrimeKG_Disease_ChemicalEntity2.csv')

  Disease Head resolved: 5,528 kept / 3,860 dropped
Saved 5,528 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/PrimeKG_Disease_ChemicalEntity2.csv


---
## Part 6 · Dropped relation types ()

In [65]:
# The following three relation types were commented out in the original notebook.
# They are intentionally skipped here. The data is available in RELWISE_DIR if needed.

# df_contraindication      — Disease_Drug: disease->drug association; Head annotation incomplete
# df_disease_phenotype_negative — Phenotype_Disease (negative): negative associations; logic incomplete
# df_anatomy_protein_absent    — Anatomy_Gene (absent): gene absent in tissue; Head annotation incomplete

print("Dropped: contraindication, disease_phenotype_negative, anatomy_protein_absent")

Dropped: contraindication, disease_phenotype_negative, anatomy_protein_absent


---
## Part 7 · Summary — all output files

In [68]:
from glob import glob
import pandas as pd, os

print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []

for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df = total_df.head(5)
        available = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/PrimeKG/



/tmp/ipykernel_1812242/3439688071.py:10: DtypeWarning: Columns (2,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  total_df = pd.read_csv(file_path)


Total files: 27


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,PriemKG_Anatomy_Gene.csv,1518203,{Anatomy_Gene},{Anatomy},{Gene},{PrimeKG},{UBERON_ID},{NCBI_Symbol}
1,PrimKG_Molfunc_Gene.csv,69530,{MolFunction_Gene},{MolFunction},{Gene},{PrimeKG},{Quick_GO},{NCBI_Symbol}
2,PrimeKG_Anatomy_Anatomy.csv,14032,{Anatomy_Anatomy},{Anatomy},{Anatomy},{PrimeKG},{UBERON},{UBERON}
3,PrimeKG_BiologicalProcess_BiologicalProcess.csv,52886,{BiologicalProcess_BiologicalProcess},{BiologicalProcess},{BiologicalProcess},{PrimeKG},{Quick_GO},{Quick_GO}
4,PrimeKG_BiologicalProcess_ChemicalEntity_1.csv,1625,{BiologicalProcess_ChemicalEntity},{BiologicalProcess},{ChemicalEntity},{PrimeKG},{QuickGO},{Pubchem}
5,PrimeKG_BiologicalProcess_Gene.csv,144805,{BiologicalProcess_Gene},{BiologicalProcess},{Gene},{PrimeKG},{Quick_GO},{NCBI_Symbol}
6,PrimeKG_CellComp_Chemical.csv,5,{CellularComponent_ChemicalEntity},{CellularComponent},{ChemicalEntity},{PrimeKG},{Quick_GO},{Pubchem_ID}
7,PrimeKG_Chemical_Chemical_1.csv,385,{ChemicalEntity_ChemicalEntity},{ChemicalEntity},{ChemicalEntity},{PrimeKG},{Pubchem},{Pubchem}
8,PrimeKG_Chemical_Chemical_2.csv,1336314,{ChemicalEntity_ChemicalEntity},{ChemicalEntity},{ChemicalEntity},{PrimeKG},{Pubchem},{Pubchem}
9,PrimeKG_Disease_Chemical.csv,1377,{Disease_ChemicalEntity},{Disease},{ChemicalEntity},{PrimeKG},{DOID_ID},{Pubchem}
